# BDB27 — geometric attribution: who was he actually covering?

**What this notebook is.** It matches each DB 30–60° game turn to **the single opposing
route-runner within 3 yd at the break frame**, then answers four questions in this order:

1. **How often was the D199 elimination rule wrong?** On turns where both rules can be
   evaluated, how often does the geometric match name a different player? This is the number
   that says whether the D199 null was attenuated by misattribution. It is reported even if
   everything else stops.
2. **How much of the band does geometry cover?** Unique, ambiguous and none at 2, 3 and 5 yd.
3. **What is the de-attenuated estimate?** The D199 estimator, unchanged, on the geometric
   unique-match set, side by side with the elimination coefficient.
4. **What happens on MAN-only coverage**, the strongest-attribution slice.

**What it is not.** It is not a causal estimate and not proof of coverage. The dataset records
no opponent identifier and no assignment, so a receiver within 3 yd at the break is only
*plausibly* the man covered. Every table below is labelled **INFERENCE**.

**How to run it.** Put the full game tracking in `data/raw/` (or set `TRACKING` in §1), open this
notebook, press **Run All**. Nothing below §1 needs editing. The run is chunked and resumable: if
it dies, Run All again and it continues from the last completed file.

**Specification.** `notes/geometric-attribution-spec.md` — committed before any coefficient was
computed on a geometric set. The estimator is `notes/outcome-test-spec.md`'s, unchanged.

## 1. Paths — the only cell to edit

* `TRACKING` — the full game tracking. `None` auto-detects `data/raw/game_tracking.csv`, else the
  three per-season files (`game_tracking_2023.csv`, `_2024`, `_2025`). A folder is accepted and
  expanded to every `*.csv` in it; a list of files is accepted. **The full file, not the sample.**
* `TURNS` — the band-turn table written by the full run. `None` = the repo's
  `data/interim/full_run_20261111/bdb27_full_db_turns.parquet`.
* `PLAYER_PLAY` — `player_play.csv`. `None` = the repo's `data/raw`.
* `OUT_DIR` — where the artefacts and the manifest go. `None` = `outputs/geometric_attribution`.
* `CHUNK_ROWS` — tracking rows per chunk. Lower it if the laptop runs short of memory; it changes
  speed, never a result.
* `RESUME` — reuse an extraction shard whose input file is unchanged.
* `ALLOW_SAMPLE_TRACKING` — **test only.** Bypasses the truncation stop and stamps everything
  `PLACEHOLDER_SAMPLE`. Leave it `False` for the real run.

In [ ]:
# ======================= CONFIG — the only cell to edit =========================
TRACKING = None          # None = auto-detect data/raw/game_tracking.csv or the season files
TURNS = None             # None = repo data/interim/full_run_20261111/bdb27_full_db_turns.parquet
PLAYER_PLAY = None       # None = repo data/raw/player_play.csv
OUT_DIR = None           # None = repo outputs/geometric_attribution
CHUNK_ROWS = 2_000_000   # rows per chunk; lower on a small-memory laptop
RESUME = True            # reuse a shard whose input file is unchanged
ALLOW_SAMPLE_TRACKING = False  # test only: bypasses the stop, stamps PLACEHOLDER_SAMPLE
# =============================================================================

## 2. Setup

Finds the repo (walks up until it sees `config.yaml` and `src/bdb27`), puts `src/` on the path and
imports `bdb27.geometric_match` — the same module the headless script uses, so the rule, the stop
conditions and the tables exist in exactly one place. The estimator is `bdb27.outcome_test`'s own
functions, not a copy: `run_all` and `verdict` are called directly.

In [ ]:
import importlib
import json
import sys
import time
from pathlib import Path


def find_repo_root() -> Path:
    '''Walk upward from the notebook's working directory.'''
    here = Path.cwd().resolve()
    for parent in (here, *here.parents):
        if (parent / "config.yaml").exists() and (parent / "src" / "bdb27").is_dir():
            return parent
    raise RuntimeError(
        "Cannot locate the BDB27 repo root (need config.yaml + src/bdb27). "
        "Open this notebook from inside the BDB27 checkout.")


REPO = find_repo_root()
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))

import pandas as pd  # noqa: E402

from bdb27 import geometric_match, outcome_test  # noqa: E402

importlib.reload(geometric_match)
importlib.reload(outcome_test)

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
print("repo     :", REPO)
print("git HEAD :", geometric_match.git_head(REPO))
print("rule     :", geometric_match.RULE_STRING)
print("status   :", geometric_match.PROVENANCE_INFERENCE)

## 3. Inputs and pre-flight — is the tracking file the full one?

The check is on **game tracking**, not on `player_play`. `player_play` holds only the 497-prospect
cohort, so a low rows-per-play there is by design (D198) and is not a defect. Full tracking is the
opposite case: it holds every player on the field, so a file with four players per play is a
truncation. The stop conditions are the spec's, and a stop names the file, the value and the
threshold.

| Check | Full-file expectation |
|---|---|
| exact row count | not 1,048,575 (2²⁰ − 1) |
| distinct players | ≥ 300 |
| median players per play | ≥ 15 |
| band plays covered | ≥ 90% |

Nothing is substituted and nothing is extracted if a stop fires.

In [ ]:
def resolve_path(value) -> Path:
    '''A user path, resolved against the repo root when it is relative.'''
    path = Path(value).expanduser()
    if not path.is_absolute():
        path = REPO / path
    return path.resolve()


raw = REPO / "data" / "raw"
turns_path = (resolve_path(TURNS) if TURNS
              else (REPO / "data" / "interim" / "full_run_20261111"
                    / "bdb27_full_db_turns.parquet"))
pp_path = (resolve_path(PLAYER_PLAY) if PLAYER_PLAY
           else (raw / "player_play.csv").resolve())
out_dir = (resolve_path(OUT_DIR) if OUT_DIR
           else (REPO / "outputs" / "geometric_attribution"))

if TRACKING is None:
    candidates = [raw / "game_tracking.csv"] + [
        raw / f"game_tracking_{year}.csv" for year in (2023, 2024, 2025)]
    tracking = [p.resolve() for p in candidates if p.exists()]
else:
    given = TRACKING if isinstance(TRACKING, (list, tuple)) else [TRACKING]
    tracking = []
    for item in given:
        p = resolve_path(item)
        if p.is_dir():
            tracking.extend(sorted(q.resolve() for q in p.glob("*.csv")
                                   if "sample" not in q.name.lower()))
        else:
            tracking.append(p)
if len(tracking) > 1:
    both = [p for p in tracking if p.name == "game_tracking.csv"]
    if both:
        tracking = both[:1]

inputs = [("turns (required)", turns_path), ("player_play (required)", pp_path)]
inputs += [(f"tracking {i + 1}", p) for i, p in enumerate(tracking)]
print(f"{'input':<24}{'present':<9}path")
for label, path in inputs:
    print(f"{label:<24}{'yes' if path.exists() else 'NO':<9}{path}")

missing = [label for label, path in inputs if not path.exists()]
if missing:
    print()
    print("STOP — these inputs are missing:", ", ".join(missing))
    if not turns_path.exists():
        print("  The band-turn table is written by the full run. Open")
        print("  notebooks/bdb27_full_run.ipynb, set MODE = 'full', Run All there, then")
        print("  return to this notebook. It looked for:", turns_path)
    if not tracking:
        print("  The notebook looked for data/raw/game_tracking.csv and")
        print("  data/raw/game_tracking_2023.csv, _2024.csv, _2025.csv.")
    raise SystemExit("missing inputs")

out_dir.mkdir(parents=True, exist_ok=True)
band = geometric_match.load_band(turns_path)
band_plays = band[["game_id", "play_id"]].drop_duplicates()
print()
print(f"band turns : {len(band):,} in {len(band_plays):,} plays, "
      f"{band['nfl_id'].nunique():,} players")

preflight_path = out_dir / "preflight.json"
cached = {}
if preflight_path.exists():
    try:
        cached = json.loads(preflight_path.read_text(encoding="utf-8"))
    except Exception:
        cached = {}

counts = {}
for p in tracking:
    key = str(p)
    stat = p.stat()
    hit = cached.get(key, {})
    if hit.get("size") == stat.st_size and hit.get("mtime_ns") == stat.st_mtime_ns:
        counts[key] = hit
    else:
        print(f"pre-flight: counting rows in {p.name} (one streaming pass)")
        counts[key] = geometric_match.preflight_counts(p, chunk_rows=CHUNK_ROWS)
        counts[key]["size"] = stat.st_size
        counts[key]["mtime_ns"] = stat.st_mtime_ns
preflight_path.write_text(json.dumps(counts, indent=2), encoding="utf-8")

verdicts = [geometric_match.preflight_verdict(c, band_plays=len(band_plays))
            for c in counts.values()]
print()
for c, v in zip(counts.values(), verdicts):
    print(f"  {Path(c['path']).name}: {c['rows']:,} rows · {c['games']:,} games · "
          f"{c['plays']:,} plays · {c['players']:,} players · median "
          f"{c['median_players_on_play']:.1f} players/play -> "
          f"{'OK' if v['ok'] else 'STOP'}")

if not all(v["ok"] for v in verdicts) and not ALLOW_SAMPLE_TRACKING:
    print()
    for v in verdicts:
        for s in v["stops"]:
            print(s)
    print()
    print("Nothing was extracted. This is the pre-flight doing its job: the geometric rule")
    print("needs every player in the frame, and this file does not hold them.")
    raise SystemExit("pre-flight stop")

if not all(v["ok"] for v in verdicts):
    print()
    print("=" * 78)
    print("WARNING — PRE-FLIGHT STOP BYPASSED (ALLOW_SAMPLE_TRACKING = True).")
    print("Every number below is PLACEHOLDER_SAMPLE and is NOT a result.")
    print("=" * 78)

stamp = "PLACEHOLDER_SAMPLE" if ALLOW_SAMPLE_TRACKING else "FULL"

## 4. Extraction — stream the tracking, keep the band's plays

The file is read in chunks and only rows for the band's `(game_id, play_id)` pairs are kept, one
Parquet shard per input file. A shard is reused when its input file's size, mtime and chunk size
match, so **Run All after a crash continues from the last completed file** instead of starting
over. The progress line prints rows read, rows kept, the share of the file and an ETA.

In [ ]:
_last = {"t": time.time()}


def progress(ev: dict) -> None:
    now = time.time()
    if ev.get("reused"):
        print(f"  {Path(ev['path']).name}: reused shard "
              f"({ev.get('rows_kept', 0):,} rows kept earlier)")
        return
    if now - _last["t"] < 2.0 and (ev.get("file_frac") or 0) < 1.0:
        return
    _last["t"] = now
    frac = ev.get("file_frac") or 0.0
    eta = ev.get("eta_s")
    eta_txt = f" · ETA {eta:,.0f}s" if eta else ""
    print(f"  {Path(ev['path']).name}: {frac:5.1%} of file · "
          f"{ev['rows_read']:,} rows read · {ev['rows_kept']:,} kept{eta_txt}")


t0 = time.time()
shards = geometric_match.extract_plays(
    tracking, band_plays, out_dir, chunk_rows=CHUNK_ROWS, resume=RESUME,
    expected_rows={str(Path(c["path"])): c["rows"] for c in counts.values()},
    progress=progress)
print()
print(f"extraction finished in {time.time() - t0:,.0f}s")
for s in shards:
    print(f"  {Path(s['path']).name:<32}{s['rows_read']:>12,} read"
          f"{s['rows_kept']:>12,} kept{'  (reused)' if s.get('reused') else ''}")
if sum(s["rows_kept"] for s in shards) == 0:
    print()
    print("STOP — no tracking row for any band play. The tracking file does not cover")
    print("this turn table at all (check the season and the game ids).")
    raise SystemExit("empty extraction")

## 5. The match — one row per turn, the frozen rule

For each turn the notebook finds the DB's own frame nearest to the turn table's
`(x_centre, y_centre)`, requires it within 0.25 yd, then looks at every tracked player in that
frame (±0.05 s). A **candidate** is an opposing-side player with a recorded route. **Exactly one
candidate inside 3 yd = attributed.** Two or more = ambiguous and dropped. None = dropped. No tie
is broken by radius, distance or row order.

Statuses: `unique` · `multiple` (ambiguous) · `none` (opponents present, none in radius) ·
`unobservable` (no opponent in the frame) · `untracked` (no tracked frames for this DB) ·
`break_frame_unrecovered` (position gate failed) · `untagged` (no team label).

In [ ]:
con = geometric_match.connect()
t0 = time.time()
bps = geometric_match.recover_break_frames(con, band, shards)
print(f"break frames: {int(bps['break_frame_ok'].sum()):,} of {len(band):,} recovered "
      f"within {geometric_match.ALIGN_GATE_YD} yd ({time.time() - t0:,.0f}s)")

frames = geometric_match.candidate_frame(con, bps, shards)
print(f"candidate frame: {len(frames):,} player-rows across "
      f"{frames.groupby(list(geometric_match.TURN_KEY), sort=False).ngroups:,} turns")

match = geometric_match.match_turns(band, frames, bps, pp_path)
match.to_csv(out_dir / "match_turns.csv", index=False, encoding="utf-8")

coverage = geometric_match.coverage_table(match, len(band))
coverage.to_csv(out_dir / "band_coverage.csv", index=False, encoding="utf-8")
print()
print(coverage[["radius_yd", "band_turns", "unique", "multiple", "none",
                "unobservable", "untracked", "break_frame_unrecovered",
                "attributed_fraction"]].to_string(index=False))

unseen = int(match["match_status"].isin(["unobservable", "untracked"]).sum())
print()
print(f"attributed at 3 yd: {int((match['match_status'] == 'unique').sum()):,} "
      f"({(match['match_status'] == 'unique').mean():.1%} of the band)")
print(f"no opposing player observed or play not tracked: {unseen:,} "
      f"({unseen / len(band):.1%})")

## 6. The misattribution rate — did the elimination rule name the right man?

D199 attached each turn to the **single visible opposing route-runner** on the play. Here the same
1,731 turns are matched geometrically. Where geometry yields a unique receiver, the two rules can
be compared:

* **agree** — same player;
* **disagree** — a different player: the elimination rule misattributed the turn;
* **ambiguous** / **no receiver in radius** — geometry cannot decide; excluded from the rate.

The rate is `disagree / (agree + disagree)`. It is printed with its denominator, and below 50
judgeable turns it is not reported at all. This number is the one that says whether attenuation
is a live explanation for the D199 null — it is not itself an estimate of the attenuation.

In [ ]:
elim = outcome_test.build_sample(geometric_match.connect(), turns_path, pp_path)
miss = geometric_match.compare_elimination(match, elim)
miss["rows"].to_csv(out_dir / "misattribution_turns.csv", index=False, encoding="utf-8")

pair = pd.DataFrame([
    {"case": "agree — same player", "turns": miss["agree"]},
    {"case": "disagree — elimination named another player", "turns": miss["disagree"]},
    {"case": "ambiguous — two or more in radius", "turns": miss["ambiguous"]},
    {"case": "no geometric receiver in radius", "turns": miss["none"]},
    {"case": "unmeasurable — not tracked / frame unrecovered", "turns": miss["unmeasurable"]},
])
print(f"elimination-attributed turns compared: {miss['elimination_turns']:,}")
print(pair.to_string(index=False))
print()
if miss["misattribution_rate"] is None:
    print(f"{miss['label']} — judgeable {miss['judgeable']:,} "
          f"(minimum {geometric_match.MIN_JUDGEABLE:,})")
else:
    print(f"{miss['label']}: {miss['misattribution_rate']:.1%} "
          f"({miss['disagree']:,} of {miss['judgeable']:,} judgeable turns)")
    print(f"ambiguous share of the elimination set: "
          f"{miss['ambiguous'] / max(miss['elimination_turns'], 1):.1%}")

## 7. The estimate — the D199 estimator, unchanged, on the geometric set

Sample: `unique` at 3 yd whose matched receiver has a `separation_at_pass_forward` value.

| Role | Variable |
|---|---|
| Outcome | `separation_at_pass_forward` (yd) on the matched receiver's row |
| Exposure | `apex_retention` |
| Controls | entry speed (linear + square), position FE, season FE, coverage-family FE |

OLS with standard errors clustered by player (CR1); player fixed effects; the nine D199
sensitivities; the seeded wild cluster bootstrap. The functions are `outcome_test.run_all` and
`outcome_test.verdict` — the same code that produced D199.

The pre-declared criterion (spec §11) is applied unchanged, and the gate (≥500 attributed turns,
≥30% of the band) decides what the number may be called. Below the gate it is
**DESCRIPTIVE ONLY**, whatever it shows.

In [ ]:
unique = match[match["match_status"] == "unique"]
sample = geometric_match.geometric_sample(match, pp_path, band)
attributed = int(len(unique))
attributed_fraction = attributed / len(band) if len(band) else 0.0
gate_met = attributed >= 500 and attributed_fraction >= 0.30

summary = {
    "provenance": stamp,
    "attribution": geometric_match.PROVENANCE_INFERENCE,
    "rule": geometric_match.RULE_STRING,
    "spec": "notes/geometric-attribution-spec.md",
    "band": {"turns": int(len(band)), "plays": int(len(band_plays))},
    "tracking": [{"file": Path(s["path"]).name, "rows": int(s["rows_read"]),
                  "rows_kept": int(s["rows_kept"])} for s in shards],
    "coverage": coverage.to_dict(orient="records"),
    "misattribution": {k: v for k, v in miss.items() if k != "rows"},
    "gate": {"attributed_turns": attributed,
             "attributed_fraction": round(attributed_fraction, 4),
             "min_turns": 500, "min_share": 0.30, "gate_met": bool(gate_met)},
    "stop": None,
}
print(f"attributed {attributed:,} ({attributed_fraction:.1%} of the band) · "
      f"with the matched receiver's separation recorded {len(sample):,} · "
      f"gate {'MET' if gate_met else 'NOT MET'}")

unseen_share = unseen / len(band) if len(band) else 0.0
if unseen_share > geometric_match.STOP_UNOBSERVED_SHARE:
    summary["stop"] = {"reason": "TRACKING DOES NOT OBSERVE OPPONENTS ON MOST PLAYS",
                       "unobservable": int(match["match_status"].eq("unobservable").sum()),
                       "untracked": int(match["match_status"].eq("untracked").sum()),
                       "band_turns": int(len(band)), "share": round(unseen_share, 4)}
    print(f"STOP — TRACKING DOES NOT OBSERVE OPPONENTS ON MOST PLAYS: {unseen:,} of "
          f"{len(band):,} ({unseen_share:.1%}) are unobservable or untracked. "
          f"That is a legitimate outcome, not a failure: this tracking file does not "
          f"hold the opposing players. No estimate is fitted.")
elif attributed == 0:
    summary["stop"] = {"reason": "NO_UNIQUE_MATCH_AT_ANY_RADIUS"}
    print("STOP — no unique match at any radius. The coverage table is the whole result.")
elif len(sample) < geometric_match.MIN_SEPARATION_TURNS:
    summary["stop"] = {"reason": "TOO_FEW_TURNS_WITH_OUTCOME",
                       "turns_with_separation": len(sample),
                       "minimum": geometric_match.MIN_SEPARATION_TURNS}
    print(f"STOP — only {len(sample):,} attributed turns carry the matched receiver's")
    print(f"separation (minimum {geometric_match.MIN_SEPARATION_TURNS:,}). No estimate is fitted.")

if summary["stop"] is None:
    summary["geometric_sample"] = dict(outcome_test.sample_summary(sample))
    fits, boot = outcome_test.run_all(sample)
    verdict = outcome_test.verdict(fits, boot)
    label = verdict["label"] if gate_met else "DESCRIPTIVE ONLY — BELOW PRE-DECLARED GATE"

    estimates = pd.DataFrame([f.as_dict() for f in fits])
    estimates.to_csv(out_dir / "estimates_geometric.csv", index=False, encoding="utf-8")
    comparison = geometric_match.compare_with_d199(
        fits, REPO / "outputs" / "outcome_test" / "estimates.csv")
    comparison["attributed_fraction_of_band"] = round(attributed_fraction, 4)
    comparison.to_csv(out_dir / "estimates_comparison.csv", index=False, encoding="utf-8")
    (out_dir / "verdict.json").write_text(json.dumps(verdict, indent=2), encoding="utf-8")

    print()
    print(comparison[["spec", "elimination_beta", "elimination_lo", "elimination_hi",
                      "elimination_n", "geometric_beta", "geometric_lo",
                      "geometric_hi", "geometric_n"]].to_string(index=False))
    print()
    print(f"primary: {fits[0].beta:+.3f} yd per unit retention "
          f"[{fits[0].lo:+.3f}, {fits[0].hi:+.3f}] · n={fits[0].n:,} · "
          f"players={fits[0].n_clusters:,} · wild-cluster p={boot['p_wild_cluster']}")
    print(f"attributed fraction of the band: {attributed_fraction:.1%} "
          f"({attributed:,} of {len(band):,})")
    print("label  :", label)
    summary["estimate"] = {
        "beta": round(float(fits[0].beta), 4), "lo": round(float(fits[0].lo), 4),
        "hi": round(float(fits[0].hi), 4), "p": round(float(fits[0].p), 5),
        "n": int(fits[0].n), "clusters": int(fits[0].n_clusters),
        "wild_cluster_p": boot["p_wild_cluster"],
        "elimination_beta": float(pd.read_csv(
            REPO / "outputs" / "outcome_test" / "estimates.csv", encoding="utf-8"
        ).iloc[0]["beta_yd_per_unit_retention"]),
        "label": label,
    }

## 8. Persistence, surviving sample and manifest

A diagnostic only: at the play's `pass_forward` frame, is the matched DB still the nearest
defender to that receiver? The pre-registered estimator has no persistence step, so this is
reported beside the estimate and never used to filter the primary sample. If the tracking has no
`pass_forward` event, the notebook says so.

In [ ]:
pf = geometric_match.pass_forward_points(con, shards)
if len(pf):
    at_pf = geometric_match.players_at_pass_forward(con, pf, shards)
    pers = geometric_match.persistence_check(match, pf, at_pf, pp_path)
    pers.to_csv(out_dir / "persistence.csv", index=False, encoding="utf-8")
    counts_p = pers["persistence"].value_counts().to_dict()
    summary["persistence"] = {str(k): int(v) for k, v in counts_p.items()}
    print("persistence at pass forward: " +
          " · ".join(f"{k} {v:,}" for k, v in sorted(counts_p.items())))
else:
    summary["persistence"] = {"status": "PERSISTENCE NOT EVALUABLE"}
    print("PERSISTENCE NOT EVALUABLE — the tracking carries no pass_forward event")

(out_dir / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
lines = geometric_match.plain_english_verdict(summary)
(out_dir / "verdict.txt").write_text("\n".join(lines) + "\n", encoding="utf-8")

written = [out_dir / "match_turns.csv", out_dir / "band_coverage.csv",
           out_dir / "misattribution_turns.csv", out_dir / "summary.json",
           out_dir / "verdict.txt"]
for extra in ("estimates_geometric.csv", "estimates_comparison.csv",
              "persistence.csv", "verdict.json"):
    if (out_dir / extra).exists():
        written.append(out_dir / extra)

manifest = geometric_match.write_manifest(
    out_dir, [turns_path, pp_path, *tracking], written,
    config={"chunk_rows": CHUNK_ROWS, "resume": RESUME,
            "allow_sample_tracking": ALLOW_SAMPLE_TRACKING},
    extra={"provenance": stamp, "stage": "complete",
           "shards": [{"file": Path(s["path"]).name,
                       "rows_read": int(s["rows_read"]),
                       "rows_kept": int(s["rows_kept"]),
                       "sha256_head": s.get("sha256_head")} for s in shards]})
print(f"artefacts: {out_dir}")
print(f"manifest : {manifest['manifest_path']} "
      f"({len(manifest['inputs'])} inputs, {len(manifest['outputs'])} outputs hashed)")

## 9. Limitations that ship with the number

1. The match is an **inference**. A receiver within 3 yd at the break is plausibly, not certainly,
   the man covered; rubs, picks and zone-match break it.
2. **Prospect versus prospect.** The matched receiver needs a `player_play` row for the outcome,
   so both sides of the pair are in the 497-player cohort.
3. Zone turns stay in the primary; MAN-only is the sensitivity.
4. Players, not turns, are the unit of independence; precision is player-clustered.
5. `separation_at_pass_forward` is measured after the route continues.
6. No causal reading: the break is not randomised.

## 10. The verdict

In [ ]:
print("\n".join(geometric_match.plain_english_verdict(summary)))
print()
print("This notebook does not upgrade a null into a claim. If the label is not a finding,")
print("the number and its interval are the result, and nothing more is asserted.")